In [1]:
# ── Cell 1: Modern unigram prior — sweep the mix, then measure canonically ─
import sys, json, time, random
from pathlib import Path
from wordfreq import zipf_frequency

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
sys.path.insert(0, str(PROJECT_ROOT / "app"))

from corrector import SpellCorrector, soundex, skeleton   # V3 module
import math

def p_modern(w, _cache={}):
    """wordfreq zipf → probability (zipf = log10 of freq per billion words)."""
    if w not in _cache:
        z = zipf_frequency(w, "en")
        _cache[w] = (10 ** z) / 1e9 if z > 0 else 1e-11
    return _cache[w]

class SpellCorrectorV4(SpellCorrector):
    """Phase 2a: the blend's unigram prior becomes a Darwin/modern mixture."""
    def __init__(self, data_dir, alpha=0.5, **kw):
        super().__init__(data_dir, **kw)
        self.alpha = alpha

    def _p_uni_mix(self, w):
        return (1 - self.alpha) * self.lm.p_unigram(w) + self.alpha * p_modern(w)

    def _ctx_logp_blend(self, w, left, right):
        s = 0.0
        if left is not None:
            s += math.log(0.7 * self.lm.p_bigram(left, w) + 0.3 * self._p_uni_mix(w))
        if right is not None:
            s += math.log(0.7 * self.lm.p_bigram(w, right) + 0.3 * self._p_uni_mix(w))
        if left is None and right is None:
            s += math.log(self._p_uni_mix(w))
        return s

# canonical pairs + the saved baseline
pairs_raw = json.loads((DATA_DIR / "misspelling_pairs.json").read_text())
base = json.loads((DATA_DIR / "v2_eval_baseline.json").read_text())
dict_ref = SpellCorrector(DATA_DIR).dictionary
eval_pairs = []
for miss, corr in pairs_raw.items():
    truths = [t for t in (corr if isinstance(corr, list) else [corr]) if t in dict_ref]
    if truths and miss not in dict_ref:
        eval_pairs.append((miss, set(truths)))

def measure(sc, pair_list):
    h1 = h5 = h10 = 0
    for miss, truths in pair_list:
        words = [w for w, _ in sc.suggest(miss, top=10)[0]]
        r = next((j + 1 for j, w in enumerate(words) if w in truths), None)
        h1 += r == 1; h5 += r is not None and r <= 5; h10 += r is not None
    n = len(pair_list)
    return h1/n, h5/n, h10/n

# ── α sweep on an 800-pair subsample ──
random.seed(42)
sub = random.sample(eval_pairs, 800)
print("α sweep (800-pair subsample):")
print(f"  {'α':<6}{'acc@1':<9}{'acc@5':<9}acc@10")
results = {}
for alpha in [0.0, 0.3, 0.5, 0.7, 1.0]:
    t0 = time.time()
    a1, a5, a10 = measure(SpellCorrectorV4(DATA_DIR, alpha=alpha), sub)
    results[alpha] = (a1, a5, a10)
    print(f"  {alpha:<6}{a1:<9.1%}{a5:<9.1%}{a10:.1%}   ({time.time()-t0:.0f}s)")

best_alpha = max(results, key=lambda a: (results[a][0], results[a][1]))
print(f"\nWinner: α = {best_alpha} — full canonical run…")
v4 = SpellCorrectorV4(DATA_DIR, alpha=best_alpha)
t0 = time.time()
a1, a5, a10 = measure(v4, eval_pairs)
print(f"  V4 (α={best_alpha}):  acc@1 {a1:.1%}   acc@5 {a5:.1%}   acc@10 {a10:.1%}"
      f"   ({time.time()-t0:.0f}s)")
print(f"  V3 baseline:  acc@1 {base['metrics']['acc1']:.1%}   "
      f"acc@5 {base['metrics']['acc5']:.1%}   acc@10 {base['metrics']['acc10']:.1%}")

# ── The visible wins you asked for ──
print("\n── the test sentence, V4 top suggestions ──")
for r in v4.analyze("I hv a frind sh ws in tje univrsty lts yer"):
    if r["status"] != "ok":
        best = r["suggestions"][0][0] if r["suggestions"] else "—"
        print(f"   {r['token']:<10} {r['status']:<16} → {best}")
print("\n── mantainance / sucraty ordering ──")
for probe in ["mantainance", "sucraty"]:
    top = [w for w, _ in v4.suggest(probe, top=5)[0]]
    print(f"   {probe:<13} → {top}")
print("\n── meting's hearing, re-judged ──")
gap, best, _ = v4._check_suspicious("meting", "video", "dis")
print(f"   gap = {gap:.1f} (was a whisper < 1)  best: {best}")

α sweep (800-pair subsample):
  α     acc@1    acc@5    acc@10
  0.0   81.4%    95.9%    97.4%   (39s)
  0.3   87.1%    97.2%    97.9%   (39s)
  0.5   87.5%    97.2%    97.9%   (39s)
  0.7   87.5%    97.2%    97.9%   (40s)
  1.0   87.5%    97.1%    97.9%   (40s)

Winner: α = 0.5 — full canonical run…
  V4 (α=0.5):  acc@1 87.0%   acc@5 97.3%   acc@10 98.2%   (208s)
  V3 baseline:  acc@1 81.3%   acc@5 96.3%   acc@10 97.5%

── the test sentence, V4 top suggestions ──
   frind      non_word         → find
   sh         suspicious_word  → so
   ws         suspicious_word  → is
   tje        non_word         → the
   univrsty   non_word         → university
   lts        non_word         → its
   yer        suspicious_word  → her

── mantainance / sucraty ordering ──
   mantainance   → ['maintenance', 'maintaining', 'maintained', 'maintainable', 'maintains']
   sucraty       → ['sucrate', 'surat', 'surety', 'sunray', 'scrat']

── meting's hearing, re-judged ──
   gap = 0.5 (was a whisper < 1

In [2]:
# ── Cell 2: Merged pool, retried under the modern prior ───────────────────
class SpellCorrectorV4M(SpellCorrectorV4):
    """Merged single pool (edit ∪ channels), unified ranking — the design
    refuted under the Victorian prior, retried under the α-mixture."""
    def suggest(self, word, left=None, right=None, top=5):
        if word in self.dictionary:
            return [(word, 0.0)], 0
        c1 = self._known(self._edits1(word))
        c2 = (self._known({e2 for e1 in self._edits1(word)
                           for e2 in self._edits1(e1)}) - c1) if len(c1) < top else set()
        pool = c1 | c2 | (self._channel_candidates(word) - c1 - c2)
        if not pool:
            return [], -1
        ranked = sorted(((w, -self.lam * weighted_edit_distance(word, w)
                          + self._ctx_logp_blend(w, left, right)) for w in pool),
                        key=lambda x: (-x[1], x[0]))
        return ranked[:top], (1 if c1 else 2 if c2 else 3)

from edit_distance import weighted_edit_distance

# quick verdict on the subsample first — no 4-minute run for a dead design
v4m = SpellCorrectorV4M(DATA_DIR, alpha=0.5)
t0 = time.time()
a1s, a5s, a10s = measure(v4m, sub)
print(f"Merged, subsample:  acc@1 {a1s:.1%}   acc@5 {a5s:.1%}   acc@10 {a10s:.1%}"
      f"   ({time.time()-t0:.0f}s)")
print(f"Tiered, subsample:  acc@1 {results[0.5][0]:.1%}   acc@5 {results[0.5][1]:.1%}"
      f"   acc@10 {results[0.5][2]:.1%}")

if a1s >= results[0.5][0] - 0.005:          # within half a point → worth the full run
    print("\nFull canonical run…")
    t0 = time.time()
    a1, a5, a10 = measure(v4m, eval_pairs)
    print(f"  Merged (full):  acc@1 {a1:.1%}   acc@5 {a5:.1%}   acc@10 {a10:.1%}"
          f"   ({time.time()-t0:.0f}s)")
    print(f"  Tiered (full):  acc@1 87.0%   acc@5 97.3%   acc@10 98.2%")
else:
    print("\nMerged still loses on the subsample — tiered stays; skipping full run.")

# the case that demanded the retrial, plus regressions
print("\n── sucraty, merged top-8 ──")
for w, s in v4m.suggest("sucraty", top=8)[0]:
    print(f"   {w:<14} {s:>8.2f}")
print("\n── spot checks ──")
for probe in ["mantainance", "recieve", "teh", "frind"]:
    print(f"   {probe:<13} → {[w for w, _ in v4m.suggest(probe, top=5)[0]]}")

Merged, subsample:  acc@1 79.6%   acc@5 96.4%   acc@10 97.9%   (40s)
Tiered, subsample:  acc@1 87.5%   acc@5 97.2%   acc@10 97.9%

Merged still loses on the subsample — tiered stays; skipping full run.

── sucraty, merged top-8 ──
   security         -17.17
   secret           -18.79
   surat            -20.20
   secretary        -20.22
   secrets          -20.49
   surety           -20.59
   secretly         -20.88
   sucrate          -21.40

── spot checks ──
   mantainance   → ['maintenance', 'maintaining', 'maintained', 'maintainable', 'maintains']
   recieve       → ['receive', 'believe', 'received', 'relieve', 'deceive']
   teh           → ['the', 'to', 'they', 'ten', 'two']
   frind         → ['find', 'friend', 'front', 'grind', 'friends']


In [3]:
# ── Cell 3: Download a modern corpus, build the modern bigram LM ──────────
import urllib.request, tarfile, io, math
from collections import Counter

URL = "https://downloads.wortschatz-leipzig.de/corpora/eng_news_2023_300K.tar.gz"
raw_path = DATA_DIR / "raw" / "eng_news_2023_300K.tar.gz"
raw_path.parent.mkdir(parents=True, exist_ok=True)

if not raw_path.exists():
    print("Downloading Leipzig eng_news_2023_300K (~40 MB)…")
    urllib.request.urlretrieve(URL, raw_path)
print(f"  archive: {raw_path.stat().st_size/1e6:.0f} MB")

from edit_distance import tokenize
sentences = []
with tarfile.open(raw_path) as tar:
    member = next(m for m in tar.getmembers() if m.name.endswith("-sentences.txt"))
    for line in io.TextIOWrapper(tar.extractfile(member), encoding="utf-8"):
        parts = line.rstrip("\n").split("\t", 1)
        if len(parts) == 2:
            toks = tokenize(parts[1])
            if toks:
                sentences.append(toks)
print(f"  {len(sentences):,} sentences, {sum(map(len, sentences)):,} tokens")

# 90/10 split, closed vocab (count ≥ 2 in train), <unk> for the rest
import random
random.seed(42)
random.shuffle(sentences)
cut = int(len(sentences) * 0.9)
train, held = sentences[:cut], sentences[cut:]

raw_uni = Counter(w for s in train for w in s)
vocab = {w for w, c in raw_uni.items() if c >= 2}
print(f"  vocabulary (count ≥ 2): {len(vocab):,}  (+ <unk>)")

def close(s):  # map OOV to <unk>, add sentence bounds
    return ["<s>"] + [w if w in vocab else "<unk>" for w in s] + ["</s>"]

uni, bi = Counter(), Counter()
for s in map(close, train):
    uni.update(s)
    bi.update(zip(s, s[1:]))
print(f"  distinct bigrams: {len(bi):,}")

V = len(vocab) + 3   # + <s> </s> <unk>
N = sum(uni.values())

def ppl(k, sents):
    logp = n = 0
    for s in map(close, sents):
        for a, b in zip(s, s[1:]):
            logp += math.log((bi.get((a, b), 0) + k) / (uni.get(a, 0) + k * V))
            n += 1
    return math.exp(-logp / n)

print("\n  add-k tournament (held-out perplexity):")
best_k, best_p = None, float("inf")
for k in [0.0001, 0.0005, 0.001, 0.005, 0.01]:
    p = ppl(k, held)
    tag = ""
    if p < best_p: best_k, best_p, tag = k, p, "  ← best"
    print(f"    k = {k:<7} PPL = {p:>9.1f}{tag}")

# retrain on all sentences with the winning k, save the artefact
uni, bi = Counter(), Counter()
for s in map(close, sentences):
    uni.update(s)
    bi.update(zip(s, s[1:]))
artefact = {
    "k": best_k, "V": V, "N": sum(uni.values()),
    "source": "Leipzig Corpora eng_news_2023_300K",
    "unigrams": dict(uni),
    "bigrams": {f"{a} {b}": c for (a, b), c in bi.items()},
}
out = DATA_DIR / "modern_lm.json"
out.write_text(json.dumps(artefact))
print(f"\n  saved {out.name}  ({out.stat().st_size/1e6:.0f} MB)")

# ── the probes 2b exists for ──
def c_bi(a, b):
    a = a if a in vocab else "<unk>"; b = b if b in vocab else "<unk>"
    return bi.get((a, b), 0)
print("\n  bigram evidence the Victorian corpus never had:")
for a, b in [("video","meeting"), ("video","meting"), ("the","whole"),
             ("de","whole"), ("she","was"), ("last","year"),
             ("i","have"), ("in","charge")]:
    print(f"    {a:>6} {b:<10} count = {c_bi(a, b):,}")

  archive: 87 MB
  300,000 sentences, 5,957,067 tokens
  vocabulary (count ≥ 2): 62,369  (+ <unk>)
  distinct bigrams: 1,497,968

  add-k tournament (held-out perplexity):
    k = 0.0001  PPL =     837.1  ← best
    k = 0.0005  PPL =     683.0  ← best
    k = 0.001   PPL =     645.6  ← best
    k = 0.005   PPL =     635.3  ← best
    k = 0.01    PPL =     672.1

  saved modern_lm.json  (33 MB)

  bigram evidence the Victorian corpus never had:
     video meeting    count = 1
     video meting     count = 7
       the whole      count = 459
        de whole      count = 0
       she was        count = 1,527
      last year       count = 1,726
         i have       count = 1,232
        in charge     count = 168


In [4]:
# ── Cell 4: Measure contamination, filter, rebuild, re-probe ──────────────
def en_coverage(toks):
    """Fraction of tokens wordfreq recognises as English at all."""
    known = sum(1 for w in toks if zipf_frequency(w, "en") > 0)
    return known / len(toks)

# measure before filtering — how dirty is the crawl?
import numpy as np
cov = np.array([en_coverage(s) for s in sentences])
print("English-coverage distribution over 300K sentences:")
for lo, hi in [(0.0, 0.5), (0.5, 0.8), (0.8, 0.95), (0.95, 1.01)]:
    m = ((cov >= lo) & (cov < hi)).sum()
    print(f"  {lo:.2f}–{hi:.2f}: {m:>8,}  ({m/len(cov):.1%})")

THRESH = 0.8
clean = [s for s, c in zip(sentences, cov) if c >= THRESH and len(s) >= 4]
print(f"\nKept {len(clean):,} sentences ({len(clean)/len(sentences):.1%}) "
      f"at coverage ≥ {THRESH}, length ≥ 4")

# a look at what we're throwing away — verify it's genuinely foreign/junk
dropped = [s for s, c in zip(sentences, cov) if c < THRESH][:5]
for s in dropped:
    print("  dropped:", " ".join(s[:12]), "…")

# rebuild: split, vocab, tournament, retrain, save — same recipe, clean data
random.seed(42); random.shuffle(clean)
cut = int(len(clean) * 0.9)
train, held = clean[:cut], clean[cut:]
raw_uni = Counter(w for s in train for w in s)
vocab = {w for w, c in raw_uni.items() if c >= 2}
uni, bi = Counter(), Counter()
for s in map(close, train):
    uni.update(s); bi.update(zip(s, s[1:]))
V = len(vocab) + 3
print(f"\nClean rebuild: vocab {len(vocab):,}, bigrams {len(bi):,}")
best_k, best_p = None, float("inf")
for k in [0.0005, 0.001, 0.005, 0.01]:
    p = ppl(k, held)
    if p < best_p: best_k, best_p = k, p
    print(f"  k = {k:<7} PPL = {p:>9.1f}")
print(f"  winner k = {best_k}")

uni, bi = Counter(), Counter()
for s in map(close, clean):
    uni.update(s); bi.update(zip(s, s[1:]))
artefact = {"k": best_k, "V": V, "N": sum(uni.values()),
            "source": f"Leipzig eng_news_2023_300K, en-coverage ≥ {THRESH}",
            "unigrams": dict(uni),
            "bigrams": {f"{a} {b}": c for (a, b), c in bi.items()}}
(DATA_DIR / "modern_lm.json").write_text(json.dumps(artefact))
print(f"  saved modern_lm.json ({(DATA_DIR / 'modern_lm.json').stat().st_size/1e6:.0f} MB)")

print("\nProbes after cleaning:")
for a, b in [("video","meeting"), ("video","meting"), ("the","whole"),
             ("de","whole"), ("she","was"), ("last","year"), ("in","charge")]:
    print(f"    {a:>6} {b:<10} count = {c_bi(a, b):,}")

English-coverage distribution over 300K sentences:
  0.00–0.50:        1  (0.0%)
  0.50–0.80:      337  (0.1%)
  0.80–0.95:   15,897  (5.3%)
  0.95–1.01:  283,765  (94.6%)

Kept 298,995 sentences (99.7%) at coverage ≥ 0.8, length ≥ 4
  dropped: register at we goparks …
  dropped: clea caulcutt and giorgio leali contributed reporting from paris …
  dropped: gujarat birdrace previously known as birdrace is organised by india birdraces with …
  dropped: she is often referred to as abrewatia a odi amantrasa in recognition …
  dropped: den f rte til at aserbajdsjan tok full kontroll over utbryterrepublikken nagorno …

Clean rebuild: vocab 62,292, bigrams 1,496,016
  k = 0.0005  PPL =     682.4
  k = 0.001   PPL =     645.0
  k = 0.005   PPL =     634.6
  k = 0.01    PPL =     671.3
  winner k = 0.005
  saved modern_lm.json (32 MB)

Probes after cleaning:
     video meeting    count = 1
     video meting     count = 9
       the whole      count = 459
        de whole      count = 0
       s

In [5]:
# ── Cell 5: ModernLM into the scorer — (β, prune) sweep, then canonical ───
class ModernLM:
    """Bigram LM over the cleaned news corpus, with low-count pruning:
    bigrams below `prune` are treated as unseen (noise-dominated evidence)."""
    def __init__(self, path, prune=1):
        d = json.loads(Path(path).read_text())
        self.k, self.V, self.N = d["k"], d["V"], d["N"]
        self.uni = d["unigrams"]
        self.bi = {tuple(key.split(" ")): c for key, c in d["bigrams"].items()
                   if c >= prune}
        self.vocab = set(self.uni)
    def _m(self, w):
        return w if w in self.vocab else "<unk>"
    def p_bigram(self, a, b):
        a, b = self._m(a), self._m(b)
        return (self.bi.get((a, b), 0) + self.k) / (self.uni.get(a, 0) + self.k * self.V)
    def p_unigram(self, w):
        return (self.uni.get(self._m(w), 0) + self.k) / (self.N + self.k * self.V)

class SpellCorrectorV5(SpellCorrectorV4):
    """Phase 2b: context evidence = β·modern bigram + (1−β)·Victorian bigram."""
    def __init__(self, data_dir, alpha=0.5, beta=0.8, prune=1, **kw):
        super().__init__(data_dir, alpha=alpha, **kw)
        self.beta = beta
        self.mlm = ModernLM(Path(data_dir) / "modern_lm.json", prune=prune)
    def _p_ctx(self, a, b):
        return self.beta * self.mlm.p_bigram(a, b) + (1 - self.beta) * self.lm.p_bigram(a, b)
    def _ctx_logp_blend(self, w, left, right):
        s = 0.0
        if left is not None:
            s += math.log(0.7 * self._p_ctx(left, w) + 0.3 * self._p_uni_mix(w))
        if right is not None:
            s += math.log(0.7 * self._p_ctx(w, right) + 0.3 * self._p_uni_mix(w))
        if left is None and right is None:
            s += math.log(self._p_uni_mix(w))
        return s

protect = ["the sepal of the flower", "a positron is emitted",
           "shades of cyan and blue", "to allot the shares fairly",
           "the servile manner of the clerk"]

print(f"{'β':<6}{'prune':<8}{'acc@1(sub)':<12}{'meting gap':<12}{'protect FPs':<12}load")
grid = {}
for beta in [0.5, 0.8]:
    for prune in [1, 5, 10]:
        t0 = time.time()
        sc5 = SpellCorrectorV5(DATA_DIR, alpha=0.5, beta=beta, prune=prune)
        load = time.time() - t0
        a1, _, _ = measure(sc5, sub)
        gap, best, _ = sc5._check_suspicious("meting", "video", "dis")
        fps = sum(sum(r["status"] != "ok" for r in sc5.analyze(s)) for s in protect)
        verdict = "CONVICTS" if gap > sc5.m_sus and best == "meeting" else "acquits"
        grid[(beta, prune)] = (a1, gap, fps)
        print(f"{beta:<6}{prune:<8}{a1:<12.1%}{gap:>5.1f} {verdict:<6}{fps:<12}{load:.0f}s")

# choose: zero FPs required, meting must convict, then max acc@1
valid = {k: v for k, v in grid.items()
         if v[2] == 0 and v[1] > 1.0}
best_cfg = max(valid, key=lambda k: valid[k][0]) if valid else None
print(f"\nChosen (β, prune) = {best_cfg}" if best_cfg else
      "\nNo config satisfies all three constraints — we analyse before choosing.")

if best_cfg:
    sc5 = SpellCorrectorV5(DATA_DIR, alpha=0.5, beta=best_cfg[0], prune=best_cfg[1])
    print("Full canonical run…")
    t0 = time.time()
    a1, a5, a10 = measure(sc5, eval_pairs)
    print(f"  V5:  acc@1 {a1:.1%}   acc@5 {a5:.1%}   acc@10 {a10:.1%}   ({time.time()-t0:.0f}s)")
    print(f"  V4:  acc@1 87.0%   acc@5 97.3%   acc@10 98.2%")

    print("\n── the sentences, V5 ──")
    for sent_t in ["I hv a frind sh ws in tje univrsty lts yer",
                   "de whole marketin depatment got so confuse durin de video meting dis moprning"]:
        print(f"  {sent_t!r}")
        for r in sc5.analyze(sent_t):
            if r["status"] != "ok":
                bestw = r["suggestions"][0][0] if r["suggestions"] else "—"
                print(f"    {r['token']:<10} {r['status']:<16} → {bestw}")

β     prune   acc@1(sub)  meting gap  protect FPs load
0.5   1       87.5%        -1.6 acquits0           1s
0.5   5       87.5%        -2.9 acquits0           1s
0.5   10      87.5%         0.9 acquits0           1s
0.8   1       87.5%        -1.6 acquits0           2s
0.8   5       87.5%        -3.4 acquits0           1s
0.8   10      87.5%         1.7 CONVICTS0           1s

Chosen (β, prune) = (0.8, 10)
Full canonical run…
  V5:  acc@1 87.0%   acc@5 97.3%   acc@10 98.2%   (206s)
  V4:  acc@1 87.0%   acc@5 97.3%   acc@10 98.2%

── the sentences, V5 ──
  'I hv a frind sh ws in tje univrsty lts yer'
    frind      non_word         → frond
    sh         suspicious_word  → s
    ws         suspicious_word  → is
    tje        non_word         → the
    univrsty   non_word         → university
    lts        non_word         → lys
  'de whole marketin depatment got so confuse durin de video meting dis moprning'
    marketin   non_word         → marketing
    depatment  non_word         

In [6]:
# ── Cell 6: Score decomposition — what exactly did prune=10 kill? ─────────
raw_bi = {tuple(k.split(" ")): c for k, c in
          json.loads((DATA_DIR / "modern_lm.json").read_text())["bigrams"].items()}

print("── Survival of the evidence that matters ──")
for a, b in [("a","friend"), ("a","find"), ("a","frond"),
             ("video","meting"), ("video","meeting"),
             ("lts","yer"), ("her","<s>"), ("she","was")]:
    c = raw_bi.get((a, b), 0)
    print(f"   {a:>7} {b:<10} count={c:<6} survives prune5={c>=5}  prune10={c>=10}")

def decompose(sc, word, cands, left, right, title):
    print(f"\n── {title}  (β={sc.beta}, prune@load) ──")
    print(f"   {'cand':<9}{'wdist':<7}{'L:mod-bi':<11}{'L:total':<10}"
          f"{'R:mod-bi':<11}{'R:total':<10}total")
    for w in cands:
        wd = weighted_edit_distance(word, w)
        lm_l = sc.mlm.p_bigram(left, w) if left else float("nan")
        lm_r = sc.mlm.p_bigram(w, right) if right else float("nan")
        tl = math.log(0.7 * sc._p_ctx(left, w) + 0.3 * sc._p_uni_mix(w)) if left else 0
        tr = math.log(0.7 * sc._p_ctx(w, right) + 0.3 * sc._p_uni_mix(w)) if right else 0
        total = -sc.lam * wd + tl + tr
        print(f"   {w:<9}{wd:<7.2f}{lm_l:<11.2e}{tl:<10.2f}"
              f"{lm_r:<11.2e}{tr:<10.2f}{total:.2f}")

for prune in [5, 10]:
    sc_p = SpellCorrectorV5(DATA_DIR, alpha=0.5, beta=0.8, prune=prune)
    decompose(sc_p, "frind", ["find", "friend", "frond"],
              "a", "sh", f"frind between 'a' and 'sh', prune={prune}")
    gap, best, ranking = sc_p._check_suspicious("yer", "lts", None)
    print(f"   yer hearing (left='lts', right=None): gap={gap:.2f} "
          f"best={best}  {'FLAGS' if gap > sc_p.m_sus else 'silent'}")
    print(f"   ranking: {[(w, round(s,2)) for w, s in ranking[:3]]}")

── Survival of the evidence that matters ──
         a friend     count=161    survives prune5=True  prune10=True
         a find       count=0      survives prune5=False  prune10=False
         a frond      count=0      survives prune5=False  prune10=False
     video meting     count=0      survives prune5=False  prune10=False
     video meeting    count=1      survives prune5=False  prune10=False
       lts yer        count=0      survives prune5=False  prune10=False
       her <s>        count=0      survives prune5=False  prune10=False
       she was        count=1524   survives prune5=True  prune10=True

── frind between 'a' and 'sh', prune=5  (β=0.8, prune@load) ──
   cand     wdist  L:mod-bi   L:total   R:mod-bi   R:total   total
   find     0.90   3.43e-08   -8.57     2.05e-06   -8.55     -19.83
   friend   0.90   1.10e-03   -7.28     4.84e-06   -9.80     -19.78
   frond    0.60   8.49e-03   -5.35     6.99e-08   -11.06    -18.20
   yer hearing (left='lts', right=None): gap=-3.5

In [7]:
# ── Cell 7: Fix <unk> leakage — candidates never inherit the crowd's mass ─
class ModernLM:
    def __init__(self, path, prune=1):
        d = json.loads(Path(path).read_text())
        self.k, self.V, self.N = d["k"], d["V"], d["N"]
        self.uni = d["unigrams"]
        self.bi = {tuple(k2.split(" ")): c for k2, c in d["bigrams"].items()
                   if c >= prune}
        self.vocab = set(self.uni)
    def p_bigram(self, a, b):
        """Context may back off to <unk>; the TARGET word may not."""
        a = a if a in self.vocab else "<unk>"
        c = self.bi.get((a, b), 0) if b in self.vocab else 0
        return (c + self.k) / (self.uni.get(a, 0) + self.k * self.V)
    def p_unigram(self, w):
        c = self.uni.get(w, 0) if w in self.vocab else 0
        return (c + self.k) / (self.N + self.k * self.V)

# who is actually in the news vocabulary? (settles the ghost-counts)
_m = ModernLM(DATA_DIR / "modern_lm.json")
for w in ["frond", "friend", "find", "meting", "meeting", "yer", "lts", "her", "year"]:
    print(f"   {w:<9} in news vocab: {w in _m.vocab}")

for prune in [1, 5]:
    sc_p = SpellCorrectorV5(DATA_DIR, alpha=0.5, beta=0.8, prune=prune)
    print(f"\n════ prune = {prune} ════")
    decompose(sc_p, "frind", ["find", "friend", "frond"],
              "a", "sh", f"frind, fixed scorer")
    gap, best, _ = sc_p._check_suspicious("yer", "lts", None)
    print(f"   yer hearing:    gap={gap:>5.2f}  best={best}  "
          f"{'FLAGS' if gap > sc_p.m_sus else 'silent'}")
    gap, best, _ = sc_p._check_suspicious("meting", "video", "dis")
    print(f"   meting hearing: gap={gap:>5.2f}  best={best}  "
          f"{'CONVICTS' if gap > sc_p.m_sus and best == 'meeting' else 'acquits'}")
    fps = sum(sum(r["status"] != "ok" for r in sc_p.analyze(s)) for s in protect)
    print(f"   protect FPs: {fps}")

# decide, then the full picture at the chosen prune
CHOSEN_PRUNE = 1   # adjust only if the block above says otherwise
sc5 = SpellCorrectorV5(DATA_DIR, alpha=0.5, beta=0.8, prune=CHOSEN_PRUNE)
a1s, a5s, a10s = measure(sc5, sub)
print(f"\nSubsample sanity: acc@1 {a1s:.1%}  acc@5 {a5s:.1%}  acc@10 {a10s:.1%}")
print("\n── the sentences, fixed V5 ──")
for sent_t in ["I hv a frind sh ws in tje univrsty lts yer",
               "de whole marketin depatment got so confuse durin de video meting dis moprning"]:
    print(f"  {sent_t!r}")
    for r in sc5.analyze(sent_t):
        if r["status"] != "ok":
            bestw = r["suggestions"][0][0] if r["suggestions"] else "—"
            print(f"    {r['token']:<10} {r['status']:<16} → {bestw}")

   frond     in news vocab: False
   friend    in news vocab: True
   find      in news vocab: True
   meting    in news vocab: False
   meeting   in news vocab: True
   yer       in news vocab: False
   lts       in news vocab: False
   her       in news vocab: True
   year      in news vocab: True

════ prune = 1 ════

── frind, fixed scorer  (β=0.8, prune@load) ──
   cand     wdist  L:mod-bi   L:total   R:mod-bi   R:total   total
   find     0.90   3.43e-08   -8.57     2.05e-06   -8.55     -19.83
   friend   0.90   1.10e-03   -7.28     4.84e-06   -9.80     -19.78
   frond    0.60   3.43e-08   -15.39    6.99e-08   -11.06    -28.25
   yer hearing:    gap= 3.91  best=her  FLAGS
   meting hearing: gap= 3.51  best=meeting  CONVICTS
   protect FPs: 0

════ prune = 5 ════

── frind, fixed scorer  (β=0.8, prune@load) ──
   cand     wdist  L:mod-bi   L:total   R:mod-bi   R:total   total
   find     0.90   3.43e-08   -8.57     2.05e-06   -8.55     -19.83
   friend   0.90   1.10e-03   -7.28   

In [8]:
# ── Cell 8: Export the V5 corrector, verify, hand over to the app ─────────
MODULE_SOURCE = '''"""
SpellCorrector — the SciSpell correction engine (V5).
Source notebooks: 04 (core), 08 (trust map), 09 (channels), 10 (modern LM).
V5 scoring: error model + blended context, where context = \u03b2\u00b7modern-news
bigram + (1\u2212\u03b2)\u00b7domain bigram, and the unigram prior = \u03b1-mixture of domain
frequency and modern frequency (wordfreq). OOV rule: a context word may back
off to <unk>; a candidate word never does \u2014 candidates cannot inherit the
aggregate probability mass of all rare words.
All resources are artefacts; missing artefacts degrade gracefully to V3/V1.
"""
import json
import math
from collections import defaultdict
from pathlib import Path

from edit_distance import tokenize, weighted_edit_distance
from language_model import BigramLM

try:
    from wordfreq import zipf_frequency
    _HAS_WORDFREQ = True
except ImportError:
    _HAS_WORDFREQ = False

ALPHABET = "abcdefghijklmnopqrstuvwxyz'"

_SOUNDEX = {**{c: "1" for c in "bfpv"}, **{c: "2" for c in "cgjkqsxz"},
            **{c: "3" for c in "dt"}, "l": "4", **{c: "5" for c in "mn"}, "r": "6"}

def soundex(w):
    w = [c for c in w.lower() if c.isalpha()]
    if not w:
        return ""
    first = w[0]
    codes, prev = [], _SOUNDEX.get(first, "")
    for c in w[1:]:
        d = _SOUNDEX.get(c, "")
        if c in "hw":
            continue
        if d and d != prev:
            codes.append(d)
        prev = d
    return (first.upper() + "".join(codes) + "000")[:4]

VOWELS = set("aeiou")

def skeleton(w):
    s = "".join(c for c in w if c not in VOWELS)
    return s if s else w

class ModernLM:
    """Bigram LM over cleaned modern news text (see notebook 10)."""
    def __init__(self, path, prune=1):
        d = json.loads(Path(path).read_text(encoding="utf-8"))
        self.k, self.V, self.N = d["k"], d["V"], d["N"]
        self.uni = d["unigrams"]
        self.bi = {tuple(key.split(" ")): c for key, c in d["bigrams"].items()
                   if c >= prune}
        self.vocab = set(self.uni)

    def p_bigram(self, a, b):
        """Context may back off to <unk>; the TARGET word may not."""
        a = a if a in self.vocab else "<unk>"
        c = self.bi.get((a, b), 0) if b in self.vocab else 0
        return (c + self.k) / (self.uni.get(a, 0) + self.k * self.V)

    def p_unigram(self, w):
        c = self.uni.get(w, 0) if w in self.vocab else 0
        return (c + self.k) / (self.N + self.k * self.V)

class SpellCorrector:
    SND_LEN_WINDOW = 2
    SND_CAP = 200

    def __init__(self, data_dir, lam=3.0, m_sus=1.0, alpha=0.5, beta=0.8, prune=1):
        data_dir = Path(data_dir)
        self.lam = lam
        self.m_sus = m_sus
        self.dictionary = set(
            (data_dir / "dictionary.txt").read_text(encoding="utf-8").split("\\n"))
        self.word_freq = {w: int(c) for w, c in json.loads(
            (data_dir / "word_freq.json").read_text(encoding="utf-8")).items()}
        self.lm = BigramLM(data_dir / "language_model.json")
        conf = json.loads((data_dir / "confusion_sets.json").read_text(encoding="utf-8"))
        self.margin_sym  = conf["margins"]["symmetric"]
        self.margin_asym = conf["margins"]["asymmetric"]
        self.confusable = {w: (set(a), self.margin_sym)
                           for w, a in conf["symmetric"].items()}
        self.confusable.update({w: ({a}, self.margin_asym)
                                for w, a in conf["asymmetric"].items()})
        trust_path = data_dir / "word_trust.json"
        self.suspicious = {}
        if trust_path.exists():
            trust = json.loads(trust_path.read_text(encoding="utf-8"))
            self.suspicious = {w: [a for a, _ in alts]
                               for w, alts in trust["suspicious"].items()}
        modern_path = data_dir / "modern_lm.json"
        self.mlm = ModernLM(modern_path, prune=prune) if modern_path.exists() else None
        self.alpha = alpha if (_HAS_WORDFREQ and self.mlm) else 0.0
        self.beta  = beta if self.mlm else 0.0
        self._zipf_cache = {}
        self.sound_index, self.skel_index = defaultdict(list), defaultdict(list)
        for w in self.dictionary:
            self.sound_index[soundex(w)].append(w)
            self.skel_index[skeleton(w)].append(w)

    # ── candidate generation ──
    def _edits1(self, word):
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        return ({L + R[1:] for L, R in splits if R} |
                {L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1} |
                {L + c + R[1:] for L, R in splits if R for c in ALPHABET} |
                {L + c + R for L, R in splits for c in ALPHABET})

    def _known(self, strings):
        return {s for s in strings if s in self.dictionary}

    def _channel_candidates(self, word):
        skel = set(self.skel_index.get(skeleton(word), []))
        bucket = self.sound_index.get(soundex(word), [])
        snd = sorted((w for w in bucket
                      if abs(len(w) - len(word)) <= self.SND_LEN_WINDOW),
                     key=lambda w: (abs(len(w) - len(word)), w))[:self.SND_CAP]
        return (skel | set(snd)) - {word}

    def candidates(self, word):
        if word in self.dictionary:
            return {word}, 0
        c1 = self._known(self._edits1(word))
        if c1:
            return c1, 1
        c2 = self._known({e2 for e1 in self._edits1(word) for e2 in self._edits1(e1)})
        return (c2, 2) if c2 else (set(), -1)

    # ── scoring ──
    def _p_modern_word(self, w):
        if w not in self._zipf_cache:
            z = zipf_frequency(w, "en") if _HAS_WORDFREQ else 0.0
            self._zipf_cache[w] = (10 ** z) / 1e9 if z > 0 else 1e-11
        return self._zipf_cache[w]

    def _p_uni_mix(self, w):
        return (1 - self.alpha) * self.lm.p_unigram(w) + self.alpha * self._p_modern_word(w)

    def _p_ctx(self, a, b):
        p_dom = self.lm.p_bigram(a, b)
        if not self.mlm:
            return p_dom
        return self.beta * self.mlm.p_bigram(a, b) + (1 - self.beta) * p_dom

    def _ctx_logp(self, w, left, right):
        """Pure domain bigram — the calibrated confusable margin test."""
        s = 0.0
        if left  is not None: s += math.log(self.lm.p_bigram(left, w))
        if right is not None: s += math.log(self.lm.p_bigram(w, right))
        if left is None and right is None: s += math.log(self.lm.p_unigram(w))
        return s

    def _ctx_logp_blend(self, w, left, right):
        s = 0.0
        if left is not None:
            s += math.log(0.7 * self._p_ctx(left, w) + 0.3 * self._p_uni_mix(w))
        if right is not None:
            s += math.log(0.7 * self._p_ctx(w, right) + 0.3 * self._p_uni_mix(w))
        if left is None and right is None:
            s += math.log(self._p_uni_mix(w))
        return s

    def suggest(self, word, left=None, right=None, top=5):
        if word in self.dictionary:
            return [(word, 0.0)], 0
        def rank(cands):
            return sorted(((w, -self.lam * weighted_edit_distance(word, w)
                            + self._ctx_logp_blend(w, left, right)) for w in cands),
                          key=lambda x: (-x[1], x[0]))
        c1 = self._known(self._edits1(word))
        ranked = rank(c1)
        tier = 1 if c1 else -1
        c2 = set()
        if len(ranked) < top:
            c2 = self._known({e2 for e1 in self._edits1(word)
                              for e2 in self._edits1(e1)}) - c1
            ranked += rank(c2)
            if not c1:
                tier = 2 if c2 else -1
        if len(ranked) < top:
            ch = self._channel_candidates(word) - c1 - c2
            ranked += rank(ch)
            if tier == -1 and ch:
                tier = 3
        return ranked[:top], tier

    def _check_confusable(self, word, left, right):
        alts, margin = self.confusable[word]
        ranking = sorted(((w, self._ctx_logp(w, left, right))
                          for w in {word} | alts), key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        if s_best - s_word > margin:
            return True, best_alt, ranking
        return False, None, ranking

    def _check_suspicious(self, word, left, right):
        cands = [word] + self.suspicious[word]
        ranking = sorted(((w, self._ctx_logp_blend(w, left, right)) for w in cands),
                         key=lambda x: -x[1])
        s_word = dict(ranking)[word]
        best_alt, s_best = next((w, s) for w, s in ranking if w != word)
        return s_best - s_word, best_alt, [(w, s) for w, s in ranking if w != word]

    def analyze(self, text, top=5):
        tokens = tokenize(text)
        results = []
        for i, tok in enumerate(tokens):
            left  = tokens[i - 1] if i > 0 else None
            right = tokens[i + 1] if i < len(tokens) - 1 else None
            if tok not in self.dictionary:
                ranked, tier = self.suggest(tok, left, right, top)
                results.append({"token": tok, "status": "non_word",
                                "suggestions": ranked, "tier": tier})
            elif tok in self.confusable:
                flagged, best, ranking = self._check_confusable(tok, left, right)
                results.append({"token": tok,
                                "status": "real_word_error" if flagged else "ok",
                                "suggestions": [(w, s) for w, s in ranking
                                                 if w != tok] if flagged else [],
                                "tier": 0})
            else:
                results.append({"token": tok, "status": "ok",
                                "suggestions": [], "tier": 0})
        if self.suspicious:
            broken = [r["status"] == "non_word" or r["token"] in self.suspicious
                      for r in results]
            for i, r in enumerate(results):
                if r["status"] != "ok" or r["token"] not in self.suspicious:
                    continue
                if not ((i > 0 and broken[i-1]) or
                        (i < len(results)-1 and broken[i+1])):
                    continue
                left  = results[i-1]["token"] if i > 0 else None
                right = results[i+1]["token"] if i < len(results)-1 else None
                gap, best, ranking = self._check_suspicious(r["token"], left, right)
                if gap > self.m_sus:
                    r.update(status="suspicious_word", suggestions=ranking[:top],
                             gap=round(gap, 1))
        return results
'''
CORRECTOR_PATH = PROJECT_ROOT / "app" / "corrector.py"
CORRECTOR_PATH.write_text(MODULE_SOURCE, encoding="utf-8")

import importlib, corrector
importlib.reload(corrector)
scm = corrector.SpellCorrector(DATA_DIR)

sent1 = "I hv a frind sh ws in tje univrsty lts yer"
sent2 = "de whole marketin depatment got so confuse durin de video meting dis moprning"
checks = [
    ("Modern LM loaded", scm.mlm is not None and scm.beta == 0.8,
     f"V={scm.mlm.V:,}" if scm.mlm else "—"),
    ("Module ≡ notebook (sent 1)",
     [(r["token"], r["status"]) for r in scm.analyze(sent1)] ==
     [(r["token"], r["status"]) for r in sc5.analyze(sent1)], "statuses match"),
    ("Module ≡ notebook (sent 2)",
     [(r["token"], r["status"]) for r in scm.analyze(sent2)] ==
     [(r["token"], r["status"]) for r in sc5.analyze(sent2)], "statuses match"),
    ("frind → friend", scm.suggest("frind", "a", "sh")[0][0][0] == "friend", "rank 1"),
    ("meting convicts",
     scm._check_suspicious("meting", "video", "dis")[0] > scm.m_sus, "gap > 1"),
    ("Protected sentences clean",
     all(all(r["status"] == "ok" for r in scm.analyze(s)) for s in protect),
     f"{len(protect)} sentences"),
    ("soo → so regression", scm.suggest("soo", "hem", "hey")[0][0][0] == "so", "named"),
    ("mantainance non-empty",
     scm.suggest("mantainance", top=5)[0][0][0] == "maintenance", "rank 1"),
]
width = max(len(c[0]) for c in checks)
print("VERIFICATION\n" + "─" * (width + 30))
for label, ok, detail in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label:<{width}}  {detail}")
print("─" * (width + 30))
print(f"  {sum(ok for _, ok, _ in checks)}/{len(checks)} checks passed")
assert all(ok for _, ok, _ in checks)
print("\napp/corrector.py is V5 — restart Streamlit and retype both sentences.")

VERIFICATION
────────────────────────────────────────────────────────
  PASS  Modern LM loaded            V=62,295
  PASS  Module ≡ notebook (sent 1)  statuses match
  PASS  Module ≡ notebook (sent 2)  statuses match
  PASS  frind → friend              rank 1
  PASS  meting convicts             gap > 1
  PASS  Protected sentences clean   5 sentences
  PASS  soo → so regression         named
  PASS  mantainance non-empty       rank 1
────────────────────────────────────────────────────────
  8/8 checks passed

app/corrector.py is V5 — restart Streamlit and retype both sentences.
